# FusionMatch — Phase 4: Vector Indexing & Bayesian Threshold Calibration

This notebook demonstrates:
1. **FAISS IndexIVFPQ Building & Compression**: Compressing 256-d vectors into ~1.2 MB index structures ($N_{\text{list}}=400, M=32, N_{\text{bits}}=8$).
2. **Baseline Exact Search Comparison**: Measuring Recall@10 against brute-force `IndexFlatIP`.
3. **nprobe Tuning**: Analyzing the Pareto frontier tradeoff of Recall@K vs query latency.
4. **Bayesian Threshold Calibration**: Calibrating category-specific decision cutoffs with Beta-Binomial posterior modeling.

In [ ]:
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss

# Ensure project root is on sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.indexing import IndexBuilder, tune_nprobe, BayesianThresholdCalibrator
from src.training.metrics import compute_pairwise_f1, compute_precision_recall_at_k
from src.utils.seed import seed_everything

seed_everything(42)
print(f"FAISS version: {faiss.__version__}")

## 1. Build IndexIVFPQ & Compare with Flat Baseline

In [ ]:
# Synthesize/Load 10,000  vectors (256-d, float32)
N = 10000
D = 256
catalog_embeddings = np.random.randn(N, D).astype(np.float32)
sku_ids = [f"B0{i:08d}" for i in range(N)]

save_dir = project_root / "artifacts" / "index"
builder = IndexBuilder(embed_dim=D, nlist=400, m=32, nbits=8)

# Build compressed IVF-PQ index
ivf_index = builder.build(catalog_embeddings, sku_ids, save_dir=save_dir, nprobe=16)

# Build exact Flat IP baseline index
flat_index = builder.build_flat_baseline(catalog_embeddings, save_dir=save_dir)

# File size comparison
ivf_size_mb = (save_dir / "index.faiss").stat().st_size / (1024 * 1024)
flat_size_mb = (save_dir / "index_flat_baseline.faiss").stat().st_size / (1024 * 1024)

print("\n=== STORAGE COMPARISON ===")
print(f"Flat Exact Index:      {flat_size_mb:.2f} MB")
print(f"Compressed IndexIVFPQ: {ivf_size_mb:.2f} MB (Compression Ratio: {flat_size_mb/ivf_size_mb:.1f}x)")

## 2. nprobe Tuning: Recall vs. Latency Tradeoff

Testing `nprobe` across a grid of $(1, 4, 8, 16, 32, 64)$ against the brute-force baseline ground truth.

In [ ]:
queries = catalog_embeddings[:200]
nprobe_results = tune_nprobe(
    index=ivf_index,
    flat_index=flat_index,
    query_embeddings=queries,
    k=10,
    nprobe_grid=(1, 4, 8, 16, 32, 64),
)

df_nprobe = pd.DataFrame(nprobe_results)
display(df_nprobe)

selected_nprobe = 16
selected_row = df_nprobe[df_nprobe.nprobe == selected_nprobe].iloc[0]
print(f"\nSelected Production Default: nprobe={selected_nprobe}")
print(f"Recall@10 vs Flat: {selected_row['recall@10']*100:.2f}%")
print(f"Query Latency:     {selected_row['latency_ms_per_query']:.3f} ms / query (Budget: < 15 ms)")

## 3. Bayesian Threshold Calibration per Product Category

In [ ]:
# Synthesize validation similarity distribution for distinct product categories
np.random.seed(42)
val_pairs_by_category = {}

categories = ["CELLULAR_PHONE_CASE", "SHOES", "GROCERY", "HOME", "CHAIR"]
for cat in categories:
    # Positive pairs with high similarity
    pos = np.random.normal(0.88, 0.05, size=150).clip(0.0, 1.0)
    # Negative pairs with low similarity
    neg = np.random.normal(0.35, 0.12, size=600).clip(0.0, 1.0)
    sims = np.concatenate([pos, neg])
    labels = np.concatenate([np.ones(150), np.zeros(600)])
    val_pairs_by_category[cat] = (sims, labels)

calibrator = BayesianThresholdCalibrator(prior_alpha=2.0, prior_beta=2.0)
calibrated_thresholds = calibrator.fit(val_pairs_by_category)

threshold_file = save_dir / "thresholds.json"
calibrator.save(threshold_file)

print("=== CALIBRATED PER-CATEGORY DECISION THRESHOLDS ===")
for cat, thresh in calibrated_thresholds.items():
    print(f"{cat:<30}: {thresh:.4f}")
print(f"\nThresholds saved to: {threshold_file}")